# smilebiz-van 테스트 노트북

SMARTRO VAN API(가이드: https://exttran.smilebiz.co.kr , 실제 호출 주소: https://extvan.smilebiz.co.kr)를
단계별로(서버 상태체크 -> 공통코드조회 -> 매출집계조회 -> 매출내역조회) 호출해 결과를 눈으로 확인하기 위한 노트북입니다.

실행 전 저장소 루트에 `.env`가 채워져 있어야 합니다 (`SMARTRO_VAN_API_KEY`).
자세한 설명은 `../README.md` 참고.

## 0. 공통 설정 (Bearer 인증 + 요청 함수)

In [ ]:
import os

import requests
from dotenv import load_dotenv

load_dotenv()

VAN_API_KEY = os.environ.get("SMARTRO_VAN_API_KEY", "").strip()
if not VAN_API_KEY:
    raise RuntimeError(
        ".env에 SMARTRO_VAN_API_KEY가 설정되지 않았습니다. "
        "저장소 루트의 .env.example을 참고해 .env를 채워주세요."
    )

VAN_BASE = "https://extvan.smilebiz.co.kr"
TIMEOUT = 15

HEADERS = {
    "Accept": "application/json",
    "Authorization": f"Bearer {VAN_API_KEY}",
}


def van_get(path: str, params: dict | None = None) -> dict:
    """VAN API를 GET으로 호출하고 JSON을 반환한다. CODE != '0000'이면 경고만 출력한다
    (호출 자체는 성공했을 수 있으므로 예외를 던지지 않고, 확인은 호출부에서 하도록 둔다)."""
    resp = requests.get(f"{VAN_BASE}{path}", headers=HEADERS, params=params, timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    if data.get("CODE") != "0000":
        print(f"⚠️ CODE={data.get('CODE')} MESSAGE={data.get('MESSAGE')}")
    return data


print("VAN_BASE:", VAN_BASE)
print("Authorization 헤더 설정됨:", bool(VAN_API_KEY))

## 1. 서버연결 상태체크

`GET /V1/common/serverChecks` — 파라미터 없음. 키가 유효하고 서버가 응답하는지만 확인합니다.

In [ ]:
check = van_get("/V1/common/serverChecks")
check

## 2. 공통코드정보조회

`GET /V1/common/getCommCodeInfo` — 파라미터 없음. `GBN`(승인구분), `HID_GBN`(카드사구분),
`ORGCOD`(카드사코드) 등 다른 API 호출 시 필요한 코드값을 확인할 수 있습니다.

In [ ]:
comm_codes = van_get("/V1/common/getCommCodeInfo")
for group in comm_codes.get("CODE_INFO", []):
    print(f"[{group['GROUP_CODE']}] {group['GROUP_NAME']}")
    for code in group.get("CODES", []):
        print(f"  {code['CODE']}: {code['NAME']}")

## 3. 매출집계 조회

`GET /V1/sales/getSalesSum` — `SDATE`/`EDATE`(승인일자, YYYYMMDD)는 필수이고,
`COMP_NO`(사업자번호)/`COMP_IDX`/`TERMID`(단말기번호)로 특정 가맹점만 좁혀 조회할 수 있습니다(옵션).

아래는 기본값으로 최근 1일치를 조회합니다. 필요하면 `SDATE`/`EDATE`를 직접 바꿔서 실행하세요.

In [ ]:
from datetime import date, timedelta

import pandas as pd

yesterday = (date.today() - timedelta(days=1)).strftime("%Y%m%d")
today = date.today().strftime("%Y%m%d")

sales_sum = van_get(
    "/V1/sales/getSalesSum",
    params={
        "SDATE": yesterday,
        "EDATE": today,
        "COMP_NO": "",
        "COMP_IDX": "",
        "TERMID": "",
    },
)
sales_sum_df = pd.DataFrame(sales_sum.get("DATA", []))
print(f"조회 기간: {yesterday} ~ {today}, 건수: {len(sales_sum_df)}")
sales_sum_df

## 4. 매출내역 조회

`GET /V1/sales/getSalesList` — `SDATE`/`EDATE`는 필수이고, `STIME`/`ETIME`(시간),
`GBN`/`HID_GBN`/`ORGCOD`(위 공통코드 참고), `CDNO`(카드번호 앞 8자리), `AUTHNO`(승인번호),
`REJEC_TYPE`(거절포함여부), `CURRPAGE`(페이지) 등을 옵션으로 좁혀 조회할 수 있습니다.

⚠️ 문서와 달리 실제 서버는 (1) 파라미터 키가 값이 비어 있어도 **요청에 아예 없으면 400 에러**를 내고,
(2) `GBN`(소계구분)은 문서엔 Optional로 나와 있지만 **비어 있으면 500 에러**가 나는 사실상 필수값입니다.
`GBN`은 결제수단 하나만 고르는 값이라, 전체 매출을 보려면 위에서 조회한 `GBN` 코드(1~8)를 순회해야 합니다.

In [ ]:
gbn_codes = [
    c["CODE"]
    for g in comm_codes.get("CODE_INFO", [])
    if g["GROUP_CODE"] == "GBN"
    for c in g["CODES"]
]

all_rows = []
for gbn in gbn_codes:
    sales_list = van_get(
        "/V1/sales/getSalesList",
        params={
            "SDATE": yesterday,
            "STIME": "",
            "EDATE": today,
            "ETIME": "",
            "COMP_NO": "",
            "COMP_IDX": "",
            "TERMID": "",
            "GBN": gbn,
            "CDNO": "",
            "AUTHNO": "",
            "HID_GBN": "",
            "ORGCOD": "",
            "REJEC_TYPE": "",
            "CURRPAGE": 1,
        },
    )
    all_rows.extend(sales_list.get("DATA", []))

sales_list_df = pd.DataFrame(all_rows)
print(f"조회 기간: {yesterday} ~ {today}, 건수: {len(sales_list_df)}")
sales_list_df.head(20)

## 5. (참고) 입금 관련 API

가이드 사이트(VAN 탭)에 아래 API가 더 있지만, 이 노트북에서는 요청 파라미터를 확인하지 않았습니다.
실제로 쓰시려면 https://exttran.smilebiz.co.kr 에서 `VAN` 탭 -> 해당 항목을 눌러 `Request` 표를 직접 확인한 뒤,
위 `van_get()` 함수에 경로와 파라미터만 맞춰서 그대로 재사용하시면 됩니다.

- 입금내역 조회(집계)
- 입금내역 조회(상세)
- 입금보류내역 조회
- 청구내역 조회